In [0]:
/*
  Script: Test Gold
  Description:
    This script validates data quality in the gold layer.
    It mainly checks for data consistency between related tables.
*/

/*
================================================================
Table: dim_customers
================================================================
*/

-- All user ids match
SELECT
*
FROM silver.crm_cust_info t1
LEFT JOIN silver.erp_cust_az12 t2
  ON t1.cst_key = t2.cid
LEFT JOIN silver.erp_loc_a101 t3
  ON t1.cst_key = t3.cid
WHERE
  t1.cst_key IS NULL OR
  t2.cid IS NULL OR
  t3.cid IS NULL;

-- All gender records match
SELECT
*
FROM silver.crm_cust_info t1
  INNER JOIN silver.erp_cust_az12 t2
    ON t1.cst_key = t2.cid
WHERE t1.cst_gndr != t2.gen;

-- No duplicate records per customer
SELECT
  cst_id,
  COUNT(*)
FROM
(
  SELECT
    cst_id,
    cst_key,
    cst_firstname,
    cst_lastname,
    cst_marital_status,
    cst_gndr,
    cst_create_date,
    bdate,
    gen,
    cntry
  FROM
  (
    SELECT
    *
    FROM silver.crm_cust_info t1
    LEFT JOIN silver.erp_cust_az12 t2
      ON t1.cst_key = t2.cid
    LEFT JOIN silver.erp_loc_a101 t3
      ON t1.cst_key = t3.cid
  )
)
GROUP BY cst_id
HAVING COUNT(*) > 1
;

/*
================================================================
Table: dim_products
================================================================
*/

-- All product category ids match
SELECT
*
FROM silver.crm_prd_info t1
LEFT JOIN silver.erp_px_cat_g1v2 t2
  ON t1.cat_id = t2.id
WHERE t2.id IS NULL
;

-- No duplicate records per product
SELECT
  product_key,
  COUNT(*)
FROM
(
  SELECT
    prd_id AS product_id,
    prd_key AS product_key,
    prd_nm AS product_name,
    cat_id AS category_id,
    cat AS category_name,
    subcat AS subcategory_name,
    maintenance AS maintenance,
    prd_cost AS product_cost, 
    prd_line AS product_line,
    prd_start_dt AS start_date
  FROM silver.crm_prd_info t1
  LEFT JOIN silver.erp_px_cat_g1v2 t2
    ON t1.prd_key = t2.id
  WHERE prd_end_dt IS NULL
) AS sub
GROUP BY product_key
HAVING COUNT(*) > 1
;

/*
================================================================
Data Model: Gold
================================================================
*/

SELECT
*
FROM gold.fact_sales t1
LEFT JOIN gold.dim_customers t2
ON t1.customer_key = t2.customer_key
WHERE t2.customer_key IS NULL
;